In [ ]:
# @title Import lib
import os
import torch
import torchvision.transforms as transform
import torch.nn as nn
import torch.nn.functional as F
import torchvision
# !pip install torchsummary
# from torchsummary import summary
from torch.autograd import Variable, Function
import matplotlib.pyplot as plt
import numpy as np
import math
import time
import copy
from types import MethodType
from collections import defaultdict

In [ ]:
# @title Hyper-parameters and settings
image_size = 224
batch_size = 20
channel_size = 3
lr = 2e-2
momentum = 0.9
weight_decay = 5e-4
lr_gamma = 1e-3
lr_decay = 0.75
min_lr = 1e-5
num_epochs = 50
num_classes = 9

# var init
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import random
seed = 5234
random.seed(seed)
torch.cuda.manual_seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
# @title Data transform
tf_source = transform.Compose([
    transform.Resize((image_size,image_size)),
    transform.ToTensor(),
    transform.Normalize(mean=[0.485,0.456,0.406],  # Normalization as default setting
                        std=[0.229,0.224,0.225],
                        )
])

tf_target = transform.Compose([
    transform.Resize((image_size,image_size)),
    transform.ToTensor(),
    transform.Normalize(mean=[0.485,0.456,0.406],  # Normalization as default setting
                        std=[0.229,0.224,0.225],
                        )
])

In [ ]:
# @title Utils
def plot_graph(history):
    fig, (ax1, ax2) = plt.subplots(1, 2)
    fig.set_figwidth(10)
    fig.suptitle("Train vs Validation")
    ax1.plot(history["train_src_acc"], label="Train_src_acc")
    ax1.plot(history["train_tar_acc"], label="Train_tar_acc")
    ax1.legend()
    ax1.set_title("Accuracy (Src/Tar)")

    ax2.plot(history["train_loss"], label="Train_loss")
    ax2.legend()
    ax2.set_title("Loss")
    fig.show()

def get_optimizer(params):
    optimizer = torch.optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay, nesterov=True)
    return optimizer

def get_lr_scheduler(optimizer):
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda epoch: (1. + lr_gamma * epoch) ** (-lr_decay))
    return scheduler

def evaluate(model, save_name, loss_fn, data_loader_test):
    path_save_cp = './cp/'
    model.load_state_dict(torch.load(path_save_cp+save_name+'.pth', weights_only=True))

    # inferencing
    valid_loss, valid_correct = 0, 0
    model.to(device).eval()
    stored_lbs = stored_preds = torch.empty(0, dtype=torch.float32).to(device)
    with torch.no_grad():
        for i, vdata in enumerate(data_loader_test):
            inp_V, label_V = vdata[0].to(device), vdata[1].to(device)
            output = model(inp_V)
            loss = loss_fn(output, label_V)
            valid_loss += loss.item()
            _, preds_v = torch.max(output, 1)
            valid_correct += (output.argmax(1) == label_V).float().sum().item()

            stored_lbs = torch.cat((stored_lbs, label_V), 0)
            stored_preds = torch.cat((stored_preds, preds_v), 0)

    print('Model test result:')
    print(f'test loss: {valid_loss / len(data_loader_test):.5f} '
          f'test acc: {valid_correct / len(data_loader_test.dataset):.5f}')

    return stored_lbs, stored_preds

# FLOPS computation
# Code from https://github.com/Eric-mingjie/rethinking-network-pruning/blob/master/imagenet/l1-norm-pruning/compute_flops.py
def print_model_param_nums(model=None):
    if model == None:
        model = torchvision.models.alexnet()
    total = sum([param.nelement() if param.requires_grad else 0 for param in model.parameters()])
    print('  + Number of params: %.4fM' % (total / 1e6))

def count_model_param_flops(model=None, input_res=224, multiply_adds=True):

    prods = {}
    def save_hook(name):
        def hook_per(self, input, output):
            prods[name] = np.prod(input[0].shape)
        return hook_per

    list_1=[]
    def simple_hook(self, input, output):
        list_1.append(np.prod(input[0].shape))
    list_2={}
    def simple_hook2(self, input, output):
        list_2['names'] = np.prod(input[0].shape)


    list_conv=[]
    def conv_hook(self, input, output):
        batch_size, input_channels, input_height, input_width = input[0].size()
        output_channels, output_height, output_width = output[0].size()

        kernel_ops = self.kernel_size[0] * self.kernel_size[1] * (self.in_channels / self.groups)
        bias_ops = 1 if self.bias is not None else 0

        params = output_channels * (kernel_ops + bias_ops)
        # flops = (kernel_ops * (2 if multiply_adds else 1) + bias_ops) * output_channels * output_height * output_width * batch_size

        num_weight_params = (self.weight.data != 0).float().sum()
        flops = (num_weight_params * (2 if multiply_adds else 1) + bias_ops * output_channels) * output_height * output_width * batch_size

        list_conv.append(flops)

    list_linear=[]
    def linear_hook(self, input, output):
        batch_size = input[0].size(0) if input[0].dim() == 2 else 1

        weight_ops = self.weight.nelement() * (2 if multiply_adds else 1)
        bias_ops = self.bias.nelement()

        flops = batch_size * (weight_ops + bias_ops)
        list_linear.append(flops)

    list_bn=[]
    def bn_hook(self, input, output):
        list_bn.append(input[0].nelement() * 2)

    list_relu=[]
    def relu_hook(self, input, output):
        list_relu.append(input[0].nelement())

    list_pooling=[]
    def pooling_hook(self, input, output):
        batch_size, input_channels, input_height, input_width = input[0].size()
        output_channels, output_height, output_width = output[0].size()

        kernel_ops = self.kernel_size * self.kernel_size
        bias_ops = 0
        params = 0
        flops = (kernel_ops + bias_ops) * output_channels * output_height * output_width * batch_size

        list_pooling.append(flops)

    list_upsample=[]

    # For bilinear upsample
    def upsample_hook(self, input, output):
        batch_size, input_channels, input_height, input_width = input[0].size()
        output_channels, output_height, output_width = output[0].size()

        flops = output_height * output_width * output_channels * batch_size * 12
        list_upsample.append(flops)

    def foo(net):
        childrens = list(net.children())
        if not childrens:
            if isinstance(net, torch.nn.Conv2d):
                net.register_forward_hook(conv_hook)
            if isinstance(net, torch.nn.Linear):
                net.register_forward_hook(linear_hook)
            if isinstance(net, torch.nn.BatchNorm2d):
                net.register_forward_hook(bn_hook)
            if isinstance(net, torch.nn.ReLU):
                net.register_forward_hook(relu_hook)
            if isinstance(net, torch.nn.MaxPool2d) or isinstance(net, torch.nn.AvgPool2d):
                net.register_forward_hook(pooling_hook)
            if isinstance(net, torch.nn.Upsample):
                net.register_forward_hook(upsample_hook)
            return
        for c in childrens:
            foo(c)

    if model == None:
        model = torchvision.models.alexnet()
    foo(model)
    input = Variable(torch.rand(3,input_res,input_res).unsqueeze(0), requires_grad = True)
    out = model(input)


    total_flops = (sum(list_conv) + sum(list_linear) + sum(list_bn) + sum(list_relu) + sum(list_pooling) + sum(list_upsample))

    print('Number of FLOPs: %.6f GFLOPs (%.2f MFLOPs)' % (total_flops / 1e9, total_flops / 1e6))

    return total_flops

class ContinuousDataloader:
    def __init__(self, data_loader):
        self.data_loader = data_loader
        self.iter = iter(data_loader)

    def __iter__(self):
        return self

    def __next__(self):
        try:
            data = next(self.iter)
        except StopIteration:
            self.iter = iter(self.data_loader)
            data = next(self.iter)
        return data

    def __len__(self):
        return len(self.data_loader)

# Train Domain adaptation

In [ ]:
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

# # Mount and unzip from directory
# from google.colab import drive
# drive.mount('./drive/')
# !unzip './drive/MyDrive/99H_datasets/Office-31.zip'
# # Upload directly
# !unzip './Office-31.zip'

domain_source = 'output_2_resized/output_2_resized'
domain_target = 'DOMAIN EXPANSION/DOMAIN EXPANSION'
dataset_path = '/kaggle/input/datasets/walankitjarak/yugioh/'
# dataset_path = './Office-31/'

Ds_source = datasets.ImageFolder(dataset_path+domain_source, transform=tf_source)
Ds_target = datasets.ImageFolder(dataset_path+domain_target, transform=tf_target)

Dl_source_set = DataLoader(Ds_source, batch_size, shuffle=True, num_workers=2, drop_last=True)
Dl_target_set = DataLoader(Ds_target, batch_size, shuffle=True, num_workers=2, drop_last=True)
Dl_eval_set = DataLoader(Ds_target, batch_size, shuffle=True, num_workers=2)

# print(len(Dl_source_set.dataset), len(Dl_target_set.dataset))
# print(len(Dl_source_set), len(Dl_target_set))

In [ ]:
# @title Train Domain Adaptation function
# https://github.com/xiaoachen98/DALN

class GradientReverseFunction(Function):
    @staticmethod
    def forward(ctx, input, coeff = 1.):
        ctx.coeff = coeff
        output = input * 1.0
        return output

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.coeff, None


class GradientReverseLayer(nn.Module):
    def __init__(self):
        super(GradientReverseLayer, self).__init__()

    def forward(self, *input):
        return GradientReverseFunction.apply(*input)


class WarmStartGradientReverseLayer(nn.Module):
    """Gradient Reverse Layer :math:`\mathcal{R}(x)` with warm start

        The forward and backward behaviours are:

        .. math::
            \mathcal{R}(x) = x,

            \dfrac{ d\mathcal{R}} {dx} = - \lambda I.

        :math:`\lambda` is initiated at :math:`lo` and is gradually changed to :math:`hi` using the following schedule:

        .. math::
            \lambda = \dfrac{2(hi-lo)}{1+\exp(- α \dfrac{i}{N})} - (hi-lo) + lo

        where :math:`i` is the iteration step.

        Args:
            alpha (float, optional): :math:`α`. Default: 1.0
            lo (float, optional): Initial value of :math:`\lambda`. Default: 0.0
            hi (float, optional): Final value of :math:`\lambda`. Default: 1.0
            max_iters (int, optional): :math:`N`. Default: 1000
            auto_step (bool, optional): If True, increase :math:`i` each time `forward` is called.
              Otherwise use function `step` to increase :math:`i`. Default: False
        """

    def __init__(self, alpha = 1.0, lo = 0.0, hi = 1.,
                 max_iters = 1000., auto_step = False):
        super(WarmStartGradientReverseLayer, self).__init__()
        self.alpha = alpha
        self.lo = lo
        self.hi = hi
        self.iter_num = 0
        self.max_iters = max_iters
        self.auto_step = auto_step

    def forward(self, input):
        """"""
        coeff = np.float32(
            2.0 * (self.hi - self.lo) / (1.0 + np.exp(-self.alpha * self.iter_num / self.max_iters))
            - (self.hi - self.lo) + self.lo
        )
        if self.auto_step:
            self.step()
        return GradientReverseFunction.apply(input, coeff)

    def step(self):
        """Increase iteration number :math:`i` by 1"""
        self.iter_num += 1

def MI(outputs_target):
    batch_size = outputs_target.size(0)
    softmax_outs_t = nn.Softmax(dim=1)(outputs_target).to(device)
    avg_softmax_outs_t = torch.sum(softmax_outs_t, dim=0) / float(batch_size)
    log_avg_softmax_outs_t = torch.log(avg_softmax_outs_t + 1e-8).to(device)
    item1 = -torch.sum(avg_softmax_outs_t * log_avg_softmax_outs_t).to(device)
    item2 = -torch.sum(softmax_outs_t * torch.log(softmax_outs_t + 1e-8)).to(device) / float(batch_size)
    return item1, item2

def pairwise_js_divergence(logits1, logits2, temp=10.0):
    """
    logits1: (N, C)
    logits2: (M, C)
    return: (N, M) JS divergence matrix
    """
    
    p = F.softmax(logits1 / temp, dim=-1)
    q = F.softmax(logits2 / temp, dim=-1)

    p = p.unsqueeze(1)   # (N,1,C)
    q = q.unsqueeze(0)   # (1,M,C)

    m = 0.5 * (p + q)

    # KLDivLoss
    js = 0.5 * (
        (p * (p / m).log()).sum(-1) +
        (q * (q / m).log()).sum(-1)
    )

    return js

def get_inter_pair_loss_vectorized(labels_s, labels_t, outputs_s, outputs_t, temp, threshold):

    softmax_t = F.softmax(outputs_t, dim=1)
    confidence_t, labels_t = softmax_t.max(dim=1)
    
    mask = confidence_t >= threshold
    
    outputs_t = outputs_t[mask]
    labels_t = labels_t[mask]
    
    if outputs_t.size(0) == 0:
        return torch.tensor(0.0, device=outputs_s.device)
    
    js_matrix = pairwise_js_divergence(outputs_s, outputs_t, temp)
    
    labels_s_expand = labels_s.unsqueeze(1)
    labels_t_expand = labels_t.unsqueeze(0)
    
    label_mask = labels_s_expand == labels_t_expand
    
    if label_mask.sum() == 0:
        return torch.tensor(0.0, device=outputs_s.device)
    
    return js_matrix[label_mask].mean()

def get_intra_pair_loss_vectorized(labels, outputs, temp):

    js_matrix = pairwise_js_divergence(outputs, outputs, temp)

    label_mask = labels.unsqueeze(0) == labels.unsqueeze(1)

    # remove diagonal
    diag_mask = ~torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    mask = label_mask & diag_mask

    if mask.sum() == 0:
        return torch.tensor(0.0, device=outputs.device)

    return js_matrix[mask].mean()

def get_PDD_loss(outs, labels, temp=1.0, threshold=0.8):
    batch_size = outs.size(0) // 2

    batch_source = outs[:batch_size]
    batch_target = outs[batch_size:]

    labels_s = labels[:batch_size]
    labels_t = labels[batch_size:]

    loss_inter = get_inter_pair_loss_vectorized(
        labels_s, labels_t, batch_source, batch_target, temp, threshold
    )

    loss_intra = get_intra_pair_loss_vectorized(
        labels_s, batch_source, temp
    )

    return (temp ** 2) * (loss_inter + loss_intra)
    
class Loss_PDD_and_NWD(nn.Module):
    def __init__(self, classifier, max_iters=1000):
        super(Loss_PDD_and_NWD, self).__init__()
        self.grl = WarmStartGradientReverseLayer(alpha=1., lo=0., hi=1., max_iters=max_iters, auto_step=True)
        self.classifier = classifier
        
    @staticmethod
    def n_discrepancy(y_s, y_t):
        pre_s, pre_t = F.softmax(y_s, dim=1), F.softmax(y_t, dim=1)
        loss = (-torch.norm(pre_t, 'nuc') + torch.norm(pre_s, 'nuc')) / y_t.shape[0]
        return loss

    def forward(self, f, labels_s):
        f_grl = self.grl(f)
        y = self.classifier(f_grl)
        y_s, y_t = y.chunk(2, dim=0)
        
        loss_nwd = self.n_discrepancy(y_s, y_t)

        pseudo_label_t = y_t.argmax(1)
        labels = torch.cat((labels_s, pseudo_label_t), dim=0)
        loss_pdd = get_PDD_loss(y, labels, temp=10., threshold=0.8)

        return loss_pdd, loss_nwd

In [ ]:
def mbv3_forward(self, x):
    f = self.features(x)
    f = self.avgpool(f)
    f = torch.flatten(f, 1)
    x = self.classifier(f)

    if self.training:
        return x, f
    else: 
        return x

def start_train_domain(model, optimizer, loss_fn, loss_pdd_nwd, lr_scheduler, 
                       num_epochs, data_loader_source, data_loader_target, 
                       nwd_lambda=1., pdd_lambda =1., MI_lambda = 0.1, save_name='domain'):
    t_0 = time.time()
    best_acc = 0.
    training_logs = {"train_loss": [], "train_src_acc": [], "train_tar_acc": []}
    for epoch in range(num_epochs):
        print(f'epochs {epoch+1:04d} / {num_epochs:04d}', end='\n============\n')
        train_loss = train_domain(model, optimizer, loss_fn, loss_pdd_nwd, lr_scheduler, 
                                  data_loader_source, data_loader_target, training_logs, epoch, 
                                  nwd_lambda=nwd_lambda, pdd_lambda=pdd_lambda, MI_lambda=MI_lambda, 
                                  show_loss_batch=True)

        if epoch % 1 == 0:
            print(f"Epochs {epoch+1}".ljust(10),
                f"train loss {training_logs['train_loss'][-1]:.5f}",
                f"train source acc {training_logs['train_src_acc'][-1]:.5f}",

                f"train target acc {training_logs['train_tar_acc'][-1]:.5f}",
                )
            print("-"*80)
    
        train_acc = training_logs['train_tar_acc'][-1]
        if train_acc > best_acc:
            best_acc = train_acc
            path_save_cp = './cp/'
            if not os.path.exists(path_save_cp): os.mkdir(path_save_cp)
            torch.save(model.state_dict(), path_save_cp+save_name+'.pth')
    print(f"Best target accuracy from training log: {max(training_logs['train_tar_acc']):.5f}")
    print("-"*80)

    t_end = time.time()-t_0
    print(f"Time consumption for net (device:{device}): {t_end} sec")
    plot_graph(training_logs)

def train_domain(model, optimizer, loss_fn, loss_pdd_nwd, lr_scheduler, 
                 data_loader_source, data_loader_target, training_logs, epoch,
                 nwd_lambda=1., pdd_lambda =1., MI_lambda = 0.1, show_loss_batch=False, n_loss_batch=5):

    train_loss, train_src_correct, train_tar_correct = 0, 0, 0
    max_batches = len(data_loader_source)
    n_show_loss = math.ceil(max_batches/n_loss_batch)
    Dl_source_iter = ContinuousDataloader(data_loader_source)
    Dl_target_iter = ContinuousDataloader(data_loader_target)
    loss_pdd_nwd.train()

    for i_batch in range(max_batches):
        current_iter = i_batch + (max_batches * epoch)
        rho = current_iter / (max_batches * num_epochs)
        model.train()
        inp_S, label_S = next(Dl_source_iter)
        inp_T, label_T = next(Dl_target_iter)

        inp_S, label_S = inp_S.to(device), label_S.to(device)
        inp_T, label_T = inp_T.to(device), label_T.to(device)

        inp = torch.cat((inp_S, inp_T), dim=0)
        optimizer.zero_grad()
        output, feature = model(inp)
        output_S, output_T = output.chunk(2, dim=0)

        # compute cross entropy loss on source domain
        cls_loss = loss_fn(output_S, label_S)
        
        # compute prediction distribution discrepancy loss between domains
        # compute nuclear-norm wasserstein discrepancy loss between domains
        loss_pdd, loss_nwd = loss_pdd_nwd(feature, label_S)
        # compute mutual information maximization loss on target domain
        MI_item1, MI_item2 = MI(output_T)
        
        # for adversarial classifier, minimize negative nwd is equal to maximize nwd
        nwd_loss = -loss_nwd * nwd_lambda
        scda_loss = -pdd_lambda * rho * loss_pdd - MI_lambda * (MI_item1 - MI_item2)
        
        # total loss
        loss = cls_loss + nwd_loss + scda_loss

        loss.backward()
        optimizer.step()
        lr_scheduler.step()

        # #output predict from net
        model.eval()
        with torch.no_grad():
            prediction = model(inp)
            prediction_S, prediction_T = prediction.chunk(2, dim=0)

        if show_loss_batch:
            if (i_batch+1) % n_show_loss == 0 or i_batch+1 == max_batches:
                print(f'[{i_batch+1}/{max_batches}] '
                      f'class loss: {cls_loss.item():.4f} '
                      f'scda loss: {scda_loss.item():.4f} '
                      f'nwd loss: {nwd_loss.item():.4f} ')

        train_loss += loss.item()
        train_src_correct += (prediction_S.argmax(1) == label_S).float().sum().item()
        train_tar_correct += (prediction_T.argmax(1) == label_T).float().sum().item()
    # save training logs
    training_logs["train_loss"].append(train_loss / max_batches)
    max_len_dataset = max_batches * batch_size
    training_logs["train_src_acc"].append(train_src_correct / max_len_dataset)
    training_logs["train_tar_acc"].append(train_tar_correct / max_len_dataset)

    return train_loss

In [ ]:
model_domain = torchvision.models.mobilenet_v3_small(weights='MobileNet_V3_Small_Weights.DEFAULT')
model_domain.classifier[-1] = torch.nn.Linear(in_features=1024, out_features=num_classes)
model_domain.forward = MethodType(mbv3_forward, model_domain)
model_domain = model_domain.to(device)
# summary(model_domain, input_size=(channel_size, image_size, image_size))
# count_model_param_flops(model=model_domain.cpu().eval(), input_res=image_size, multiply_adds=True)

model_domain = model_domain.to(device)

# Optimizer and cost function
optimizer = get_optimizer(model_domain.parameters())
lr_scheduler = get_lr_scheduler(optimizer)
loss_fn = torch.nn.CrossEntropyLoss()
classifier = model_domain.classifier
pdd_nwd = Loss_PDD_and_NWD(classifier, max_iters=500).to(device)


start_train_domain(model_domain, optimizer, loss_fn, pdd_nwd, lr_scheduler, num_epochs, Dl_source_set, Dl_target_set, 
                   nwd_lambda=1., pdd_lambda =1., MI_lambda = 0.12, save_name='digital_to_irl')

model_eval = torchvision.models.mobilenet_v3_small()
model_eval.classifier[-1] = torch.nn.Linear(in_features=1024, out_features=num_classes)
model_eval = model_eval.to(device)
stored_lbs, stored_preds = evaluate(model_eval, 'digital_to_irl', loss_fn, Dl_eval_set)

In [ ]:
# model_domain = torchvision.models.mobilenet_v3_small(weights='MobileNet_V3_Small_Weights.DEFAULT')
# model_domain.classifier[-1] = torch.nn.Linear(in_features=1024, out_features=num_classes)
# model_domain = model_domain.to(device)

# print(model_domain)
# print("+++++++++++++++++++++")
# classifier = model_domain.classifier
# model_domain.forward = MethodType(mbv3_forward, model_domain)
# print(classifier)


In [ ]:
# Helper function for inline image display
def matplotlib_imshow(img, one_channel=False):
    img = img.cpu()
    if one_channel:
        img = img.mean(dim=0)
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    if one_channel:
        plt.imshow(npimg)
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))

# Evaluation result with target images and classes prediction
classes = ("effect", "fusion", "link", "normal", "ritual",  
           "spell", "synchro", "trap", "xyz")

Dl_eval_iter = iter(Dl_eval_set)
inp_S, label_S = next(Dl_eval_iter)
out = model_eval(inp_S.to(device))
img_grid = torchvision.utils.make_grid(inp_S)
matplotlib_imshow(img_grid, one_channel=True)
print(', '.join(classes[label_S[j]] for j in range(64)))
print(', '.join(classes[out.argmax(1)[j]] for j in range(64)))